# Credit Manager - Database Initialization Test
Este notebook inicializa la base de datos SQLite y verifica que todas las tablas mapeadas en SQLAlchemy se hayan creado correctamente a partir de nuestro módulo `src/database`.

In [ ]:
"""
Notebook Cell: Database Reset & Initialization
Description: Drops all existing tables and recreates them to ensure schema consistency.
Author: Juan Martín Carini
Date: 2026-05-11
"""

import sys
import os

# Asegurar que el path apunte a la raíz para encontrar el paquete 'src'
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.database import Base, engine

def reset_database():
    print("Iniciando reset de base de datos...")
    
    # 1. Eliminar todas las tablas existentes
    Base.metadata.drop_all(bind=engine)
    print("✅ Todas las tablas han sido eliminadas.")
    
    # 2. Crear las tablas con la estructura actualizada (incluyendo dia_vencimiento_default)
    Base.metadata.create_all(bind=engine)
    print("✅ Estructura de tablas recreada correctamente.")

if __name__ == "__main__":
    reset_database()

## Test de Originación y Amortización
En esta prueba vamos a:
1. Crear un **Socio Comercial** y una **Cartera**.
2. Dar de alta un **Cliente** de prueba.
3. Originar un **Crédito** con una TNA que incluya IVA.
4. Generar las **Cuotas** automáticamente usando nuestro `AmortizationEngine`.

In [ ]:
"""
Notebook Cell: Final Logic Integration Test
Description: Validates amortization calculation and dynamic 'origen' property.
Author: Juan Martín Carini
Date: 2026-05-11
"""

from datetime import date
import pandas as pd
from src.database import SessionLocal, SocioComercial, Cartera, Cliente, Credito, Cuota, SexoEnum
from src.logic.amortization import AmortizationEngine

db = SessionLocal()

try:
    # 1. Preparar Entidades de prueba
    socio = SocioComercial(razon_social="Mutual del Sur", cuit="30777888991", dia_corte=13)
    db.add(socio)
    db.flush()

    # Escenario: Crédito COMPRADO (asignado a una cartera)
    cartera = Cartera(nombre="Fideicomiso Mayo 2026", socio_id=socio.id, fecha_compra=date.today(), tna_descuento=0.45)
    db.add(cartera)
    db.flush()

    cliente = Cliente(cuil="20445556661", documento="44555666", apellido="Martínez", nombre="Luis", sexo=SexoEnum.MASCULINO)
    db.add(cliente)
    db.flush()

    # 2. Originar Crédito (Simulamos una compra de cartera)
    # Al asignar cartera_id, la propiedad @hybrid_property debe devolver COMPRADO
    nuevo_credito = Credito(
        cliente_cuil=cliente.cuil,
        cartera_id=cartera.id,
        socio_originador_id=socio.id,
        capital=150000.0,
        tna_c_iva=0.75,
        plazo=12,
        fecha_emision=date.today(),
    )
    db.add(nuevo_credito)
    db.flush()

    # 3. Validar Propiedad Dinámica antes de persistir cuotas
    print(f"ID Crédito: {nuevo_credito.id}")
    print(f"Origen detectado (Hybrid Property): {nuevo_credito.origen.value}") # Debería ser COMPRADO

    # 4. Generar Cuotas con el motor de lógica (numpy_financial)
    cuotas = AmortizationEngine.generate_french_schedule(
        credito_id=nuevo_credito.id,
        capital=nuevo_credito.capital,
        tna_c_iva=nuevo_credito.tna_c_iva,
        plazo=nuevo_credito.plazo,
        gracia=2,
        fecha_emision=nuevo_credito.fecha_emision,
        dia_corte=socio.dia_corte
    )
    db.add_all(cuotas)
    db.commit()

    # 5. Visualización de Resultados
    # Verificamos que las cuotas vencen el 28 y los montos son razonables
    query = f"SELECT * FROM cuotas WHERE credito_id = {nuevo_credito.id}"
    df_resultado = pd.read_sql(query, db.bind)
    
    # Añadimos una columna de control para ver la cuota total (PMT)
    df_resultado['total_pmt'] = df_resultado['capital'] + df_resultado['interes'] + df_resultado['iva_interes']
    
    display(df_resultado[['numero_cuota', 'fecha_vencimiento', 'capital', 'interes', 'iva_interes', 'total_pmt']])

except Exception as e:
    db.rollback()
    print(f"❌ Error en el test: {e}")
finally:
    db.close()